# SpaCal-Depth: GPU caching on Colab

Runtime → Change runtime type → **A100 GPU** (T4 / L4 also work: set `AMP = True`). TPUs (v5e / v6e) do **not** work: the code is PyTorch-CUDA.

Before the first session: upload `spacal_code.tgz` to `MyDrive/spacal/`.

Caches are written to Google Drive, so after a disconnect just Run all again: finished images are skipped.
Then: Runtime → Run all.

In [ ]:
# 1. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration
import os
DATASETS = ['nyu', 'kitti', 'diode_indoor', 'diode_outdoor']   # all in one A100 session; trim to resume selectively
AMP = False              # A100: full precision (faithful to published models). T4/L4: True (~3x faster)
NO_GRAD = False          # True skips gradient uncertainty (slowest part)
PRUNE_AFTER = False      # True deletes raw caches after tables are built (only if Drive is short of space)

DRIVE = '/content/drive/MyDrive/spacal'
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.environ.update({
    'SPACAL_DATA_ROOT': '/content/data',            # fast local disk; re-downloaded each session
    'SPACAL_CACHE_ROOT': f'{DRIVE}/cache',          # persistent -> resumable
    'SPACAL_PREPARED_ROOT': f'{DRIVE}/prepared',
    'SPACAL_RESULTS_ROOT': f'{DRIVE}/results',
    'HF_HOME': '/content/hf_cache',                 # model weights: local, re-downloaded in ~1-2 min
    'SPACAL_DEVICE': 'cuda',
    'PYTHONUNBUFFERED': '1',
})
DL_TARGETS = sorted({'nyu': 'nyu', 'kitti': 'kitti', 'diode_indoor': 'diode', 'diode_outdoor': 'diode'}[d] for d in DATASETS)
ENV = ('AMP=1 ' if AMP else '') + ('NO_GRAD=1 ' if NO_GRAD else '')
print('datasets:', DATASETS, '| downloads:', DL_TARGETS, '| env:', ENV or '(defaults)')

In [ ]:
# 3. Code (re-extracted each session so an updated tarball is picked up)
!rm -rf /content/CV_Project && tar xzf {DRIVE}/spacal_code.tgz -C /content
%cd /content/CV_Project
!ls

In [ ]:
# 4. Dependencies (Colab's own torch is kept)
!pip install -q -r requirements-colab.txt
!pip install -q --no-deps git+https://github.com/lpiccinelli-eth/UniDepth.git
!python -c "import torch, transformers, unidepth; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| unidepth ok')"

In [ ]:
# 5. Data (local disk, ~8 GB for all three sources, ~15-30 min)
!bash scripts/download_data.sh {' '.join(DL_TARGETS)}

In [ ]:
# 6. Smoke test: 3 images per (dataset, model). Look for FAILED and at the img/s.
for ds in DATASETS:
    !{ENV} LIMIT=3 bash scripts/cache_dataset.sh {ds}
!python -m experiments.e0_base_metrics --datasets {' '.join(DATASETS)}

In [ ]:
# 7. Full caching, dataset by dataset (hours; keep the tab open). Safe to re-run after a disconnect.
for ds in DATASETS:
    !{ENV} bash scripts/cache_dataset.sh {ds} 2>&1 | tee -a {DRIVE}/logs/cache_{ds}.log

In [ ]:
# 8. Status: a dataset is COMPLETE when all 4 models finished a full (non-smoke) run
import glob, json
complete = []
for ds in ['nyu', 'kitti', 'diode_indoor', 'diode_outdoor']:
    root = f"{os.environ['SPACAL_CACHE_ROOT']}/{ds}"
    if not os.path.isdir(root):
        print(f'{ds:14s} not started'); continue
    counts = {t: len(glob.glob(f'{root}/{t}/*.npz')) for t in sorted(os.listdir(root)) if os.path.isdir(f'{root}/{t}')}
    mans = [json.load(open(m)) for m in glob.glob(f'{root}/*/manifest.json')]
    done = len(mans) == 4 and all('finished' in m and not m['args']['limit'] and m['n_cached'] >= m['n_samples'] for m in mans)
    complete += [ds] if done else []
    print(f'{ds:14s}', counts, 'COMPLETE' if done else '', '(PRUNED)' if os.path.exists(f'{root}/PRUNED') else '')
print('complete:', complete)
!du -sh {DRIVE}/cache {DRIVE}/prepared 2>/dev/null

In [ ]:
# 9. Finish completed datasets: E0 metrics -> compact PixelTables -> optional prune
if 'complete' not in globals():
    raise SystemExit('run cell 8 (status) first; it defines `complete`')
ready = [d for d in complete if d in DATASETS]
if ready:
    !python -m experiments.e0_base_metrics --datasets {' '.join(ready)}
    !python -m experiments.prepare_tables --all --datasets {' '.join(ready)}
    if PRUNE_AFTER:
        for ds in ready:
            !python -m scripts.prune_cache --dataset {ds} --yes
else:
    print('nothing complete yet')
!ls -la {DRIVE}/prepared; cat {DRIVE}/results/e0/summary.md

## After all four datasets are finished

**Disconnect the GPU runtime** (it burns compute units while idle). Download `MyDrive/spacal/prepared/` and `MyDrive/spacal/results/` (a few GB) into `CV_Project/` on the laptop or the cluster and run `bash scripts/run_all_cpu.sh` there. For the qualitative figures also copy `MyDrive/spacal/cache/` (or just its RGB samples if pruned).